# 🔧 Lab 04 — dbt & BigQuery : Pipeline Medallion Bronze → Silver → Gold

**Objectifs pédagogiques :**
- Configurer dbt localement dans VS Code pour BigQuery
- Créer un pipeline de transformation complet (Bronze / Silver / Gold)
- Comprendre les matérialisations, le `ref()`, et le lineage
- Exécuter les tests de qualité de données

**Durée estimée :** 60 min  
**Environnement :** VS Code + Terminal intégré + Extension BigQuery VS Code

---

## 🛠️ Prérequis VS Code

Installez ces extensions avant de commencer :

| Extension | ID | Utilité |
|-----------|-----|---------|
| **SQLTools** | `mtxr.sqltools` | Exécuter des requêtes SQL BigQuery |
| **SQLTools BigQuery** | `Evidence.sqltools-bigquery-driver` | Driver BigQuery pour SQLTools |
| **dbt Power User** | `innoverio.vscode-dbt-power-user` | Lineage, autocomplétion dbt |

> 💡 **Convention :**  
> - 🟦 Cellules `bash` → Terminal intégré VS Code (`Ctrl+\``)  
> - 🟨 Cellules `sql` → Extension SQLTools (clic droit → *Run Query*) **ou** Console BigQuery  
> - 🟩 Cellules `yaml` → Contenu à coller dans le fichier indiqué


---
## ⚙️ Partie 0 — Authentification & Environnement local

### 0.1 — Authentifier gcloud et Application Default Credentials (ADC)

dbt utilise les **ADC** (Application Default Credentials) pour se connecter à BigQuery sans clé de service.  
Ouvrez le terminal VS Code (`Ctrl+\``) et exécutez :


In [ ]:
# Étape 1 : Se connecter à son compte Google
gcloud auth login

# Étape 2 : Créer les credentials locaux pour les SDK (dbt, Python, etc.)
gcloud auth application-default login

# Étape 3 : Définir le projet GCP actif
export PROJECT_ID="VOTRE_PROJECT_ID"   # ← remplacer ici
gcloud config set project $PROJECT_ID

echo "✅ Authentification OK"
echo "PROJECT_ID = $(gcloud config get-value project)"


### 0.2 — Créer les datasets BigQuery


In [ ]:
export PROJECT_ID=$(gcloud config get-value project)

for DATASET in bronze_landing bronze silver gold; do
  bq mk --location=EU --dataset "${PROJECT_ID}:${DATASET}"
  echo "✅ Dataset créé : ${DATASET}"
done


### 0.3 — Créer les données sources (table brute simulée)

Exécutez cette requête via **SQLTools** (clic droit → *Run Query*) ou la Console BigQuery.


In [ ]:
-- Créer la table source brute
CREATE OR REPLACE TABLE `bronze_landing.native_raw_events` (
  event_id    STRING,
  timestamp   STRING,
  user_id     STRING,
  event_type  STRING,
  device      STRUCT<os STRING, browser STRING>,
  payload     STRING
);

INSERT INTO `bronze_landing.native_raw_events` VALUES
  ('evt-001', '2024-01-15T10:23:45Z', 'usr-42', 'page_view',
      STRUCT('Android', 'Chrome'), '{}'),
  ('evt-002', '2024-01-15T10:25:01Z', 'usr-42', 'purchase_success',
      STRUCT('Android', 'Chrome'), '{"amount":49.99}'),
  ('evt-003', '2024-01-15T11:00:00Z', 'usr-17', 'page_view',
      STRUCT('iOS', 'Safari'),  '{}'),
  ('evt-004', '2024-01-15T11:05:22Z', 'usr-17', 'page_view',
      STRUCT('iOS', 'Safari'),  '{}'),
  ('evt-005', '2024-01-16T09:10:00Z', 'usr-99', 'purchase_success',
      STRUCT('Windows', 'Edge'), '{"amount":129.00}'),
  -- Doublon intentionnel sur evt-001 pour tester la déduplication Silver :
  ('evt-001', '2024-01-15T10:24:00Z', 'usr-42', 'page_view',
      STRUCT('Android', 'Chrome'), '{}');


In [ ]:
# Vérifier que la table source contient bien 6 lignes (dont 1 doublon)
bq query --nouse_legacy_sql   "SELECT COUNT(*) AS total FROM \`bronze_landing.native_raw_events\`"


---
## Partie 1 — Installation et initialisation de dbt

### 1.1 — Installer dbt-bigquery

> ⚠️ Si votre projet utilise un virtualenv Python, activez-le d'abord !


In [ ]:
# Installer dbt et le connecteur BigQuery
uv add dbt-bigquery

# Vérifier l'installation
dbt --version


### 1.2 — Initialiser le projet dbt dans le dossier du lab

Nous initialisons le projet **directement dans le dossier** `04_dbt_project/` déjà présent dans le repo.


In [ ]:
# Se placer dans le dossier du lab (adapter le chemin si besoin)
cd "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project"

# Initialiser dbt (crée les sous-dossiers manquants sans écraser les fichiers existants)
dbt init dbt_bigdata_lasalle --skip-profile-setup 2>/dev/null || true

# Créer la structure de dossiers models
mkdir -p models/bronze models/silver models/gold

echo "✅ Projet dbt initialisé"
ls -la


### 1.3 — Configurer `dbt_project.yml`

Le fichier `dbt_project.yml` est déjà présent dans `04_dbt_project/`. Vérifiez qu'il contient bien :


In [ ]:
# 04_dbt_project/dbt_project.yml
name: 'dbt_bigdata_lasalle'
version: '1.0.0'
config-version: 2

profile: 'dbt_bigdata_lasalle'

model-paths: ["models"]
test-paths: ["tests"]
macro-paths: ["macros"]

clean-targets:
  - "target"
  - "dbt_packages"

models:
  dbt_bigdata_lasalle:
    bronze:
      +materialized: view
      +schema: bronze
    silver:
      +materialized: table
      +schema: silver
    gold:
      +materialized: table
      +schema: gold


### 1.4 — Créer `profiles.yml` (connexion BigQuery locale)

Ce fichier se place dans `~/.dbt/profiles.yml` (hors du repo Git — ne jamais le committer !).


In [ ]:
# ~/.dbt/profiles.yml  ← PAS dans le repo Git
dbt_bigdata_lasalle:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth          # utilise les ADC créés à l'étape 0.1
      project: "VOTRE_PROJECT_ID"
      dataset: dbt_dev
      threads: 4
      timeout_seconds: 300
      location: EU


In [ ]:
# Créer le fichier profiles.yml automatiquement
export PROJECT_ID=$(gcloud config get-value project)
mkdir -p ~/.dbt

cat > ~/.dbt/profiles.yml << EOF
dbt_bigdata_lasalle:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: "${PROJECT_ID}"
      dataset: dbt_dev
      threads: 4
      timeout_seconds: 300
      location: EU
EOF

echo "✅ ~/.dbt/profiles.yml créé pour project=${PROJECT_ID}"


### 1.5 — Tester la connexion

✅ Si vous voyez `Connection test: [OK]`, vous êtes prêt pour la suite.


In [ ]:
cd "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project"
dbt debug


---
## 🥉 Partie 2 — Modèle Bronze : Vue de normalisation

**Principe :** Le modèle Bronze ne fait qu'exposer la table source avec un typage minimal.  
**Matérialisation :** `view` — recalculé à la demande, sans stockage supplémentaire.

### 2.1 — Créer `models/bronze/stg_raw_events.sql`


In [ ]:
-- 04_dbt_project/models/bronze/stg_raw_events.sql
-- Modèle Bronze : Normalisation minimale des événements bruts
WITH source AS (
    SELECT * FROM {{ target.project }}.bronze_landing.native_raw_events
)
SELECT
    event_id,
    CAST(timestamp AS TIMESTAMP) AS event_timestamp,
    user_id,
    event_type,
    device.os      AS device_os,
    device.browser AS device_browser,
    payload
FROM source


In [ ]:
cat > "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project/models/bronze/stg_raw_events.sql" << 'EOF'
-- Modèle Bronze : Normalisation minimale des événements bruts
WITH source AS (
    SELECT * FROM {{ target.project }}.bronze_landing.native_raw_events
)
SELECT
    event_id,
    CAST(timestamp AS TIMESTAMP) AS event_timestamp,
    user_id,
    event_type,
    device.os      AS device_os,
    device.browser AS device_browser,
    payload
FROM source
EOF
echo "✅ stg_raw_events.sql écrit"


### 2.2 — Exécuter et vérifier le modèle Bronze


In [ ]:
cd "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project"
dbt run --select stg_raw_events


In [ ]:
-- Vérifier la vue Bronze créée par dbt
-- dbt nomme le dataset : <dataset_profiles>_bronze → dbt_dev_bronze
SELECT * FROM `dbt_dev_bronze.stg_raw_events` LIMIT 10;


---
## 🥈 Partie 3 — Modèle Silver : Nettoyage & Déduplication

**Principe :** Le modèle Silver consomme le Bronze via `ref()` et déduplique les événements.  
**Matérialisation :** `table` — stockage physique dans BigQuery.

### 3.1 — Créer `models/silver/int_user_sessions.sql`


In [ ]:
-- 04_dbt_project/models/silver/int_user_sessions.sql
-- Modèle Silver : Nettoyage, déduplication et enrichissement
WITH stg AS (
    SELECT * FROM {{ ref('stg_raw_events') }}
),
dedup AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY event_id
            ORDER BY event_timestamp DESC
        ) AS rn
    FROM stg
)
SELECT
    event_id,
    event_timestamp,
    DATE(event_timestamp) AS event_date,
    user_id,
    event_type,
    device_os,
    device_browser
FROM dedup
WHERE rn = 1   -- garder uniquement l'occurrence la plus récente


In [ ]:
cat > "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project/models/silver/int_user_sessions.sql" << 'EOF'
-- Modèle Silver : Nettoyage, déduplication et enrichissement
WITH stg AS (
    SELECT * FROM {{ ref('stg_raw_events') }}
),
dedup AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY event_id
            ORDER BY event_timestamp DESC
        ) AS rn
    FROM stg
)
SELECT
    event_id,
    event_timestamp,
    DATE(event_timestamp) AS event_date,
    user_id,
    event_type,
    device_os,
    device_browser
FROM dedup
WHERE rn = 1
EOF
echo "✅ int_user_sessions.sql écrit"


### 3.2 — Exécuter et vérifier le modèle Silver


In [ ]:
cd "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project"
dbt run --select int_user_sessions


In [ ]:
-- Point de contrôle : 6 lignes source → 5 lignes après déduplication de evt-001
SELECT
    event_id,
    event_timestamp,
    user_id,
    event_type
FROM `dbt_dev_silver.int_user_sessions`
ORDER BY event_timestamp;


In [ ]:
# Compter les lignes — attendu : 5
bq query --nouse_legacy_sql   "SELECT COUNT(*) AS lignes_apres_dedup FROM \`dbt_dev_silver.int_user_sessions\`"


---
## 🥇 Partie 4 — Modèle Gold : Agrégation BI

**Principe :** Le modèle Gold agrège les données Silver pour les dashboards.  
**Matérialisation :** `table` — optimisée pour la lecture BI.

### 4.1 — Créer `models/gold/fct_daily_sales.sql`


In [ ]:
-- 04_dbt_project/models/gold/fct_daily_sales.sql
-- Modèle Gold : KPIs quotidiens par OS — prêt pour Looker Studio
WITH silver_events AS (
    SELECT * FROM {{ ref('int_user_sessions') }}
)
SELECT
    event_date,
    device_os,
    COUNT(DISTINCT user_id)
        AS active_users,
    COUNT(CASE WHEN event_type = 'page_view'        THEN 1 END)
        AS total_page_views,
    COUNT(CASE WHEN event_type = 'purchase_success' THEN 1 END)
        AS total_purchases
FROM silver_events
GROUP BY 1, 2
ORDER BY 1 DESC, 3 DESC


In [ ]:
cat > "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project/models/gold/fct_daily_sales.sql" << 'EOF'
-- Modèle Gold : KPIs quotidiens par OS — prêt pour Looker Studio
WITH silver_events AS (
    SELECT * FROM {{ ref('int_user_sessions') }}
)
SELECT
    event_date,
    device_os,
    COUNT(DISTINCT user_id)                                       AS active_users,
    COUNT(CASE WHEN event_type = 'page_view'        THEN 1 END)  AS total_page_views,
    COUNT(CASE WHEN event_type = 'purchase_success' THEN 1 END)  AS total_purchases
FROM silver_events
GROUP BY 1, 2
ORDER BY 1 DESC, 3 DESC
EOF
echo "✅ fct_daily_sales.sql écrit"


### 4.2 — Exécuter le pipeline complet en une commande


In [ ]:
cd "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project"

# Lance les 3 modèles dans l'ordre du lineage automatiquement
dbt run

# Résultat attendu :
# 1 of 3 OK created sql view model  dbt_dev_bronze.stg_raw_events      [OK]
# 2 of 3 OK created sql table model dbt_dev_silver.int_user_sessions    [OK]
# 3 of 3 OK created sql table model dbt_dev_gold.fct_daily_sales        [OK]


In [ ]:
-- Résultat Gold final — prêt pour Looker Studio
SELECT *
FROM `dbt_dev_gold.fct_daily_sales`
ORDER BY event_date DESC;


---
## 🧪 Partie 5 — Tests de qualité de données

### 5.1 — Créer `models/silver/schema.yml`


In [ ]:
# 04_dbt_project/models/silver/schema.yml
version: 2

models:
  - name: int_user_sessions
    description: "Événements nettoyés et dédupliqués — couche Silver"
    columns:
      - name: event_id
        description: "Identifiant unique de l'événement"
        tests:
          - not_null
          - unique              # vérifie que la déduplication a bien fonctionné

      - name: user_id
        description: "Identifiant de l'utilisateur"
        tests:
          - not_null

      - name: event_type
        description: "Type d'événement métier"
        tests:
          - not_null
          - accepted_values:
              values: ['page_view', 'purchase_success', 'add_to_cart', 'search']

      - name: event_date
        description: "Date de l'événement (partitionnement)"
        tests:
          - not_null


In [ ]:
cat > "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project/models/silver/schema.yml" << 'EOF'
version: 2

models:
  - name: int_user_sessions
    description: "Événements nettoyés et dédupliqués — couche Silver"
    columns:
      - name: event_id
        tests:
          - not_null
          - unique
      - name: user_id
        tests:
          - not_null
      - name: event_type
        tests:
          - not_null
          - accepted_values:
              values: ['page_view', 'purchase_success', 'add_to_cart', 'search']
      - name: event_date
        tests:
          - not_null
EOF
echo "✅ schema.yml écrit"


---
## 🔗 Partie 6 — Lineage & Documentation

### 6.1 — Générer et explorer le lineage dans VS Code

Avec l'extension **dbt Power User** installée, le lineage est visible directement dans VS Code :

1. Ouvrez n'importe quel fichier `.sql` du projet (ex: `fct_daily_sales.sql`)
2. Cliquez sur l'onglet **Lineage** dans le panneau dbt Power User (sidebar)
3. Vous verrez le graphe :

```
bronze_landing.native_raw_events
           │
           ▼
  stg_raw_events  (bronze / view)
           │
           ▼
  int_user_sessions  (silver / table)
           │
           ▼
  fct_daily_sales  (gold / table)
```

### 6.2 — Générer la documentation HTML


In [ ]:
cd "$(git rev-parse --show-toplevel)/big_data/02_cours/day2_transformation/labs/04_dbt_project"

# Générer le catalogue de données
dbt docs generate

# Ouvrir le serveur de documentation dans le navigateur
dbt docs serve --port 8080
# → Ouvrir http://localhost:8080 dans votre navigateur
# → Cliquez sur le bouton "Lineage Graph" en bas à droite


### 6.3 — Commandes dbt utiles en développement


In [ ]:
# ── Référence rapide des commandes dbt ──────────────────────────────────────

# Exécuter un modèle + toutes ses dépendances amont (+avant)
dbt run --select +fct_daily_sales

# Exécuter un modèle + tous ses descendants (après+)
dbt run --select stg_raw_events+

# Exécuter les 3 étapes : run + test + docs en une commande
dbt build

# Inspecter la requête SQL compilée (avec les ref() résolus)
dbt compile --select fct_daily_sales
cat target/compiled/dbt_bigdata_lasalle/models/gold/fct_daily_sales.sql

# Nettoyer les artefacts compilés
dbt clean


---
## ✅ Récapitulatif

| Couche | Modèle dbt | Matérialisation | Dataset BigQuery |
|--------|-----------|-----------------|-----------------|
| Source | `bronze_landing.native_raw_events` | Table native | `bronze_landing` |
| 🥉 Bronze | `stg_raw_events` | **View** | `dbt_dev_bronze` |
| 🥈 Silver | `int_user_sessions` | **Table** | `dbt_dev_silver` |
| 🥇 Gold | `fct_daily_sales` | **Table** | `dbt_dev_gold` |

### 🎯 Ce que vous avez appris

- ✅ Authentification locale avec `gcloud auth application-default login`
- ✅ La fonction `ref()` crée automatiquement le lineage entre modèles
- ✅ Les matérialisations (`view`, `table`) se configurent par couche dans `dbt_project.yml`
- ✅ Les tests YAML (`not_null`, `unique`, `accepted_values`) valident la qualité des données
- ✅ `dbt build` = `dbt run` + `dbt test` en une seule commande

### 🧠 Questions de réflexion

1. Pourquoi la déduplication se fait-elle en Silver et non en Bronze ?
2. Que se passerait-il si on changeait la matérialisation de Silver de `table` à `view` ?
3. Comment ajouter un 4ème modèle Gold `fct_revenue_by_user` qui agrège le chiffre d'affaires par utilisateur ?
